# IV_04 — Pipeline Python: el pegamento del ecosistema

## 1. Objetivo

Ejecutar un pipeline completo: leer export PI → limpiar → almacenar → consultar → exportar para Power BI.

## 2. Concepto

Python no reemplaza Excel ni Power BI. **Los conecta:**
1. Extrae datos del historiador (CSV/API)
2. Valida calidad y transforma
3. Carga en Supabase
4. Genera agregados para dashboards

In [ ]:
import os
import sqlite3
from pathlib import Path

import pandas as pd

MOD_DIR = Path.cwd()
os.chdir(MOD_DIR)
DATA_DIR = MOD_DIR / "data"
OUTPUT_DIR = MOD_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
DB_PATH = DATA_DIR / "ecosistema_local.db"

def init_local_db():
    """Carga CSV en SQLite local (fallback sin Supabase)."""
    if DB_PATH.exists():
        return
    conn = sqlite3.connect(DB_PATH)
    pd.read_csv(DATA_DIR / "equipos.csv").to_sql("equipos", conn, if_exists="replace", index=False)
    pd.read_csv(DATA_DIR / "lecturas_pi_export.csv", parse_dates=["timestamp"]).to_sql(
        "lecturas_pi", conn, if_exists="replace", index=False
    )
    pd.read_csv(DATA_DIR / "eventos_mantenimiento.csv", parse_dates=["inicio", "fin"]).to_sql(
        "eventos_mantenimiento", conn, if_exists="replace", index=False
    )
    conn.close()

def get_supabase_client():
    from dotenv import load_dotenv
    load_dotenv(MOD_DIR / ".env")
    url, key = os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY")
    if url and key:
        from supabase import create_client
        return create_client(url, key)
    return None

def query_sql(sql, params=()):
    init_local_db()
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query(sql, conn, params=params)
    conn.close()
    return df

from datetime import datetime


## 3. Paso 1 — Leer y limpiar export PI

In [ ]:
raw = pd.read_csv(DATA_DIR / "lecturas_pi_historiador.csv", parse_dates=["Timestamp"])
print(f"Registros brutos: {len(raw)}")

df_clean = raw[raw["Quality"] == "GOOD"].copy()
df_clean = df_clean.dropna(subset=["Value"])
print(f"Registros GOOD: {len(df_clean)}")
df_clean.head()


## 4. Paso 2 — Transformar para almacenamiento

In [ ]:
staging = df_clean.rename(columns={
    "Timestamp": "timestamp", "Tag": "tag", "Value": "valor",
    "Unit": "unidad", "Quality": "quality",
})
staging["timestamp"] = staging["timestamp"].astype(str)
staging = staging[["timestamp", "tag", "valor", "unidad", "quality"]]
staging.head()


## 5. Paso 3 — Consultar datos almacenados

In [ ]:
resumen = query_sql("""
    SELECT tag,
           AVG(valor) AS promedio,
           MAX(valor) AS maximo,
           COUNT(*) AS n
    FROM lecturas_pi
    WHERE quality = 'GOOD'
    GROUP BY tag
""")
resumen


## 6. Paso 4 — Exportar para Power BI

In [ ]:
# Agregado diario para dashboard
agg = (
    df_clean.set_index("Timestamp")
    .groupby("Tag")["Value"]
    .resample("1D")
    .mean()
    .reset_index()
)
agg_pbi = agg.rename(columns={"Timestamp": "Fecha", "Tag": "Tag", "Value": "Valor"})
agg_pbi["Fecha"] = agg_pbi["Fecha"].dt.strftime("%Y-%m-%d")

out_pbi = OUTPUT_DIR / "pipeline_dashboard.csv"
agg_pbi.to_csv(out_pbi, index=False)

out_resumen = OUTPUT_DIR / "pipeline_resumen_tags.csv"
resumen.to_csv(out_resumen, index=False)

print(f"Exportado Power BI: {out_pbi}")
print(f"Exportado resumen: {out_resumen}")


## 7. Interpretación para mantenimiento

Un pipeline automatizado reduce errores de copiar/pegar entre Excel y reportes. La capa SQL (Supabase) permite que operaciones, mantenimiento y gerencia consulten los mismos datos con una sola fuente de verdad.

## 8. Resumen y siguiente paso

- Pipeline: extraer → transformar → cargar → visualizar.
- Python es el pegamento entre PI, SQL y Power BI.
- Módulo B completo: ecosistema de datos Minería 5.0.

**Siguiente módulo:** `05_ia_predictiva/IV_05_mantenimiento_predictivo.ipynb`